In [ ]:
# STEP1: For Install and import libraries

!pip install -q tensorflow pandas numpy scikit-learn matplotlib seaborn pillow

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("TensorFlow version:", tf.__version__)

In [ ]:
# STEP2: Upload Metadata CSV file
from google.colab import files

uploaded = files.upload()

csv_name = list(uploaded.keys())[0]

df = pd.read_csv(csv_name)

print("CSV loaded successfully!")
print("CSV file:", csv_name)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

In [ ]:
# STEP3: Check Dataset
print("Dataset shape:", df.shape)

print("\nLabel distribution:")
print(df["Label"].value_counts())

if "split" in df.columns:
    print("\nSplit distribution:")
    print(df["split"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# STEP4: Clean Column name
df.columns = df.columns.str.strip()

print(df.columns.tolist())

In [ ]:
# STEP 5: Convert metadata to numeric(0 or 1)
metadata_columns = [
    "Fever",
    "Cough",
    "Red_eyes",
    "Koplik_spots",
    "Vaccinated"
]

for col in metadata_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing metadata with 0
df[metadata_columns] = df[metadata_columns].fillna(0)

print(df[metadata_columns].head())

In [ ]:
# STEP 6: Convert labels

df["label"] = df["Label"].astype(str).str.lower().str.strip()

label_map = {
    "normal": 0,
    "measles": 1
}

df["target"] = df["label"].map(label_map)

print(df[["label", "target"]].head())

print("\nTarget distribution:")
print(df["target"].value_counts())

In [ ]:
# STEP 7: Upload image zip file

from google.colab import files

print("Upload yourimage dataset ZIP file")

uploaded_zip = files.upload()

zip_name = list(uploaded_zip.keys())[0]

print("Uploaded:", zip_name)

In [ ]:
# Extract Dataset

import zipfile

extract_path = "/content/measles_dataset"

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:")
print(extract_path)

In [ ]:
# STEP 8: Find Image Folder

for root, dirs, files_list in os.walk(extract_path):

    if files_list:
        image_files = [
            f for f in files_list
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        ]

        if image_files:
            print(root, "->", len(image_files), "images")

In [ ]:
# STEP 9: Create image path

image_extensions = [".jpg", ".jpeg", ".png", ".webp"]

image_path_dictionary = {}

for root, dirs, files_list in os.walk(extract_path):

    for file in files_list:

        if file.lower().endswith(tuple(image_extensions)):

            image_path_dictionary[file] = os.path.join(root, file)

print("Total images found:", len(image_path_dictionary))

In [ ]:
#STEP 10: Match CSV images with actual images

def find_image_path(image_name):

    image_name = os.path.basename(str(image_name))

    if image_name in image_path_dictionary:
        return image_path_dictionary[image_name]

    # Try without extension
    base_name = os.path.splitext(image_name)[0]

    for filename, path in image_path_dictionary.items():

        if os.path.splitext(filename)[0] == base_name:
            return path

    return None


df["image_path"] = df["image_name"].apply(find_image_path)

print("Images matched:", df["image_path"].notna().sum())
print("Images missing:", df["image_path"].isna().sum())

In [ ]:
# STEP 11: Connect to Roboflow and download images
!pip install -q roboflow
import roboflow

API_KEY = "******************"
WORKSPACE = "roksanas-workspace-lsgn2"
PROJECT_ID = "final-project-dataset-yy2qw"
VERSION = 2

rf = roboflow.Roboflow(api_key=API_KEY)
workspace = rf.workspace(WORKSPACE)
project = workspace.project(PROJECT_ID)
version = project.version(VERSION)

# Used "coco" format instead of "folder"
dataset = version.download("coco")

IMAGES_ROOT = dataset.location
print("\n Dataset downloaded to:", IMAGES_ROOT)
print(os.listdir(IMAGES_ROOT))

In [ ]:
#STEP12:Re-match image_name of CSV file with images of downloaded folder
import os
import re

def normalize_name(name):
    name = name.lower()
    name = re.split(r"\.rf\.", name)[0]
    name = re.sub(r"(_jpg|_jpeg|_png|\.jpg|\.jpeg|\.png)$", "", name)
    name = re.sub(r"[^a-z0-9]", "", name)
    return name

# scan all files of train/valid/test folders and make a lookup dictionary
file_lookup = {}

for split_folder in ["train", "valid", "test"]:
    folder_path = os.path.join(IMAGES_ROOT, split_folder)   # IMAGES_ROOT = dataset.location
    if not os.path.isdir(folder_path):
        continue
    for fname in os.listdir(folder_path):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            key = normalize_name(fname)
            file_lookup[key] = os.path.join(folder_path, fname)

print("Total file index:", len(file_lookup))

def find_image_path(row):
    csv_name = str(row["image_name"])
    key = normalize_name(csv_name)
    return file_lookup.get(key, None)

df["image_path"] = df.apply(find_image_path, axis=1)

missing = df["image_path"].isna().sum()
matched = len(df) - missing
print(f"\n Matched: {matched} / {len(df)}")
print(f"Not Matched: {missing} / {len(df)}")

if missing > 0:
    print("\n Example of unmatched images:")
    print(df[df["image_path"].isna()]["image_name"].head(10).tolist())

df = df.dropna(subset=["image_path"]).reset_index(drop=True)
print("\nFinal usable rows:", len(df))
print(df[["image_name", "image_path"]].head())

In [ ]:
# STEP13: Remove missing images

df = df.dropna(subset=["image_path"]).reset_index(drop=True)

print("Final usable dataset:", len(df))

In [ ]:
# STEP 14: Use Splits

if "split" not in df.columns:

    raise ValueError(
        "Your CSV does not contain a 'split' column. "
        "Please add train/valid/test information to the CSV."
    )

df["split"] = df["split"].astype(str).str.lower().str.strip()

# Handle common naming differences
df["split"] = df["split"].replace({
    "validation": "valid",
    "val": "valid"
})

train_df = df[df["split"] == "train"].copy()
valid_df = df[df["split"] == "valid"].copy()
test_df  = df[df["split"] == "test"].copy()

print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("Test:", len(test_df))

In [ ]:
# STEP 15: Check label distributions

print("TRAIN")
print(train_df["label"].value_counts())

print("\nVALIDATION")
print(valid_df["label"].value_counts())

print("\nTEST")
print(test_df["label"].value_counts())

In [ ]:
# STEP 16: Dataset settings

IMG_SIZE = 224
BATCH_SIZE = 32

AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
#STEP 17: Image + metadataloder
def load_image(path):

    image = tf.io.read_file(path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32) / 255.0

    return image

In [ ]:
# STEP 18: Create Multimodal dataset

def create_dataset(dataframe, shuffle=False):

    image_paths = dataframe["image_path"].values

    metadata = dataframe[metadata_columns].values.astype("float32")

    labels = dataframe["target"].values.astype("float32")

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            {
                "image_input": image_paths,
                "metadata_input": metadata
            },
            labels
        )
    )

    def process_data(inputs, label):

        image = load_image(inputs["image_input"])

        metadata_values = inputs["metadata_input"]

        return (
            {
                "image_input": image,
                "metadata_input": metadata_values
            },
            label
        )

    dataset = dataset.map(
        process_data,
        num_parallel_calls=AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

In [ ]:
train_dataset = create_dataset(train_df, shuffle=True)

valid_dataset = create_dataset(valid_df, shuffle=False)

test_dataset = create_dataset(test_df, shuffle=False)

print("Datasets created successfully!")

In [ ]:
# STEP 19: BUILD MULTIMODAL CNN
# IMAGE BRANCH

image_input = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3),
    name="image_input"
)

base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_tensor=image_input
)

# Freeze pretrained layers initially
base_model.trainable = False

x = base_model.output

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(
    128,
    activation="relu"
)(x)

x = layers.Dropout(0.3)(x)

# METADATA BRANCH

metadata_input = layers.Input(
    shape=(5,),
    name="metadata_input"
)

m = layers.Dense(
    32,
    activation="relu"
)(metadata_input)

m = layers.Dropout(0.2)(m)

m = layers.Dense(
    16,
    activation="relu"
)(m)

# FUSION

combined = layers.Concatenate()([x, m])

combined = layers.Dense(
    64,
    activation="relu"
)(combined)

combined = layers.Dropout(0.3)(combined)

# OUTPUT

output = layers.Dense(
    1,
    activation="sigmoid",
    name="output"
)(combined)

# FINAL MODEL


model = Model(
    inputs=[
        image_input,
        metadata_input
    ],
    outputs=output
)

model.summary()

In [ ]:
# STEP 20: Complete Model

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy"
    ]
)

print("Model compiled successfully!")

In [ ]:
# STEP 21:Train Model

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

history = model.fit(
    train_dataset,

    validation_data=valid_dataset,

    epochs=20,

    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

In [ ]:
#STEP 22: Traning and Validation accuracy
import matplotlib.pyplot as plt

# `history` is the object returned by model.fit(...)
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs = range(len(train_acc))

# Overall (final epoch) accuracy
final_train = train_acc[-1] * 100
final_val = val_acc[-1] * 100

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs, train_acc, label='Training Accuracy')
ax.plot(epochs, val_acc, label='Validation Accuracy')

ax.set_title('Training and Validation Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend(loc='center right')

# Overall accuracy box in the bottom-right corner
ax.text(0.97, 0.04,
        f"Overall Accuracy\nTraining: {final_train:.2f}%\nValidation: {final_val:.2f}%",
        transform=ax.transAxes, ha='right', va='bottom', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.9))

plt.tight_layout()
plt.show()

In [ ]:
# STEP 23: Traning and validation loss

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")

plt.legend()
plt.show()

In [ ]:
# STEP 24: TEST EVALUATION

test_loss, test_accuracy = model.evaluate(
    test_dataset,
    verbose=1
)

print("\n====================================")
print("TEST RESULTS")
print("====================================")

print(
    f"Test Loss     : {test_loss:.4f}"
)

print(
    f"Test Accuracy : {test_accuracy * 100:.2f}%"
)

In [ ]:
# STEP 25: FINAL CLASSIFICATION METRICS

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

print("\n==========================================")
print("FINAL TEST METRICS")
print("==========================================")

print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")

In [ ]:
#STEP 26: Show overall performance
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
# Generate predictions using the trained CNN model
y_pred_prob = model.predict(test_generator)

# Convert probabilities into class predictions
y_pred = np.argmax(y_pred_prob, axis=1)

# Get actual labels
y_true = test_generator.classes

# Calculate actual performance metrics
accuracy = accuracy_score(y_true, y_pred) * 100
precision = precision_score(y_true, y_pred, average='weighted') * 100
recall = recall_score(y_true, y_pred, average='weighted') * 100
f1 = f1_score(y_true, y_pred, average='weighted') * 100

# Display actual results
print("Final Test Performance")
print("------------------------------")
print(f"Accuracy  : {accuracy:.2f}%")
print(f"Precision : {precision:.2f}%")
print(f"Recall    : {recall:.2f}%")
print(f"F1-Score  : {f1:.2f}%")

In [ ]:
# STEP 27: PREDICTIONS

y_true = test_df["target"].values

probabilities = model.predict(
    test_dataset
).ravel()

y_pred = (
    probabilities >= 0.5
).astype(int)

print("Predictions generated!")

In [ ]:
# STEP 28: CLASSIFICATION REPORT
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Normal",
            "Measles"
        ],
        zero_division=0
    )
)

In [ ]:
# STEP 29: CONFUSION MATRIX
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Normal", "Measles"],
    yticklabels=["Normal", "Measles"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
# STEP 30: SAVE MODEL

model.save(
    "/content/measles_multimodal_model.keras"
)

print("Model saved successfully!")

In [ ]:
# DOWNLOAD MODEL

from google.colab import files

files.download(
    "/content/measles_multimodal_model.keras"
)

In [ ]:
# STEP 31: UPLOAD NEW IMAGE

from google.colab import files

uploaded_test = files.upload()

new_image_name = list(uploaded_test.keys())[0]

new_image_path = "/content/" + new_image_name

print("Image:", new_image_name)

In [ ]:
# STEP 32: ENTER METADATA

print("Enter metadata values.")
print("Use 1 = Yes and 0 = No")

fever = int(input("Fever (1/0): "))
cough = int(input("Cough (1/0): "))
red_eyes = int(input("Red eyes (1/0): "))
koplik_spots = int(input("Koplik spots (1/0): "))
vaccinated = int(input("Vaccinated (1/0): "))

In [ ]:
# STEP 33: MULTIMODAL PREDICTION

# Load image
image = load_image(
    tf.constant(new_image_path)
)

image = tf.expand_dims(
    image,
    axis=0
)


# Create metadata array
metadata = np.array(
    [[
        fever,
        cough,
        red_eyes,
        koplik_spots,
        vaccinated
    ]],
    dtype=np.float32
)


# Predict
probability = model.predict(
    {
        "image_input": image,
        "metadata_input": metadata
    }
)[0][0]


if probability >= 0.5:

    prediction = "MEASLES"

    confidence = probability * 100

else:

    prediction = "NORMAL"

    confidence = (1 - probability) * 100


print("\n======================================")
print("DIAGNOSIS RESULT")
print("======================================")

print("Prediction :", prediction)
print(f"Confidence : {confidence:.2f}%")

In [ ]:
# STEP 34: DISPLAY RESULT

plt.figure(figsize=(6, 6))

img = Image.open(new_image_path)

plt.imshow(img)

plt.axis("off")

plt.title(
    f"Prediction: {prediction}\n"
    f"Confidence: {confidence:.2f}%"
)

plt.show()